In [ ]:
import requests
import pandas as pd
from datetime import datetime, date, timezone, timedelta
import json
import time
import random
from typing import Any
import json
from pathlib import Path
import re
import xml.etree.ElementTree as ET

from renewables_permitting.utils import as_list, save_parquet, validate_required_columns, clean_text
import xml.etree.ElementTree as ET

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

BOE_DOCS_XML_DIR = BRONZE_DIR / "boe_docs_xml"

BOE_CANDIDATES_PATH = SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"

BOE_CANDIDATES_DOCS_TEXT_DIR = SILVER_DIR / "boe_candidates_docs_text"

BOE_CANDIDATES_DOCS_TEXT_PATH = BOE_CANDIDATES_DOCS_TEXT_DIR / "boe_candidates_docs_text.parquet"

In [ ]:
REQUIRED_XML_DOWNLOAD_COLS = {"identificador", "doc_file_stem", "url_xml"}

REQUIRED_DOCS_TEXT_COLS = [
    "identificador",
    "doc_file_stem",
    "url_html",
    "url_xml",
    "fecha_publicacion",
    "titulo",
    "epigrafe_nombre",
    "departamento_nombre",
    "seccion_nombre",
    "xml_path",
    "texto_limpio",
    "texto_len",
    "parsed_at",
]

In [ ]:
test = pd.read_parquet(
    SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"
)

test.columns

# Funciones

Código incremental. Se añaden nuevas filas y las ya procesadas ok se quedan tal cual.

In [ ]:
def parse_boe_xml_to_text(xml_path: Path) -> str:
    """
    Extrae el contenido textual del XML del BOE conservando:

    - el orden del texto;
    - el texto situado después de elementos hijos;
    - la separación entre títulos, párrafos y filas;
    - una representación sencilla de las tablas.
    """

    tree = ET.parse(xml_path)
    root = tree.getroot()

    block_tags = {
        "p",
        "parrafo",
        "titulo",
        "subtitulo",
        "epigrafe",
        "apartado",
        "seccion",
        "section",
        "div",
        "h1",
        "h2",
        "h3",
        "h4",
        "h5",
        "h6",
        "li",
        "item",
        "blockquote",
        "tr",
        "row",
        "fila",
        "br",
    }

    cell_tags = {
        "td",
        "th",
        "cell",
        "entry",
        "celda",
    }

    text_parts: list[str] = []

    def local_name(tag: object) -> str:
        """
        Elimina el namespace XML.

        Ejemplo:
        {namespace}p -> p
        """

        if not isinstance(tag, str):
            return ""

        return tag.rsplit("}", maxsplit=1)[-1].lower()

    def append_text(value: str | None) -> None:
        """
        Añade texto conservando los espacios necesarios entre palabras.
        """

        if not value:
            return

        normalized = re.sub(r"\s+", " ", value)

        if normalized.strip():
            text_parts.append(normalized)

    def append_line_break() -> None:
        """
        Añade un salto de línea sin duplicarlo.
        """

        if text_parts and text_parts[-1] != "\n":
            text_parts.append("\n")

    def walk(element: ET.Element) -> None:
        """
        Recorre recursivamente el XML conservando el orden de text y tail.
        """

        tag = local_name(element.tag)

        if tag in block_tags:
            append_line_break()

        # Texto contenido antes del primer hijo.
        append_text(element.text)

        for child in element:
            walk(child)

            # Texto situado después del elemento hijo.
            append_text(child.tail)

        if tag in cell_tags:
            text_parts.append(" | ")
        elif tag in block_tags:
            append_line_break()

    walk(root)

    return clean_text(
        "".join(text_parts),
        preserve_line_breaks=True,
    )

In [ ]:
def get_existing_ids(output_path: Path) -> set[str]:
    if not output_path.exists():
        return set()

    existing = pd.read_parquet(output_path)

    if "identificador" not in existing.columns:
        raise ValueError("El parquet existente no contiene la columna 'identificador'.")

    return set(existing["identificador"].dropna().astype(str))

In [ ]:
def build_boe_candidates_docs_text(
    candidates: pd.DataFrame,
    xml_dir: Path,
    output_path: Path,
) -> pd.DataFrame:
    """
    Construye o actualiza la capa silver `boe_candidates_docs_text`.

    Conserva todos los candidatos BOE y añade el texto parseado del XML
    cuando está disponible.

    Estados posibles:
    - ok: XML encontrado y parseado correctamente.
    - missing: XML no encontrado en disco.
    - parse_error: XML encontrado, pero no se pudo parsear.

    La función es incremental:
    - no reprocesa registros con xml_status == "ok";
    - reprocesa registros nuevos;
    - reprocesa registros previously missing o parse_error.
    """
    required_cols = {
        "identificador",
        "doc_file_stem",
        "url_html",
        "url_xml",
        "fecha_publicacion",
        "titulo",
        "epigrafe_nombre",
        "departamento_nombre",
        "seccion_nombre",
    }

    output_cols = [
        "identificador",
        "doc_file_stem",
        "url_html",
        "url_xml",
        "fecha_publicacion",
        "titulo",
        "epigrafe_nombre",
        "departamento_nombre",
        "seccion_nombre",
        "xml_path",
        "texto_limpio",
        "texto_len",
        "xml_status",
        "parse_error",
        "parsed_at",
    ]

    output_path.parent.mkdir(parents=True, exist_ok=True)

    if not xml_dir.exists():
        raise FileNotFoundError(f"El directorio de XML no existe: {xml_dir}")

    validate_required_columns(candidates, required_cols)

    candidates = candidates.copy()
    candidates["identificador"] = candidates["identificador"].astype(str)
    candidates["doc_file_stem"] = candidates["doc_file_stem"].astype(str)

    if output_path.exists():
        existing_df = pd.read_parquet(output_path)

        if "identificador" not in existing_df.columns:
            raise ValueError(
                "El parquet existente no contiene la columna 'identificador'."
            )

        if "xml_status" not in existing_df.columns:
            existing_df["xml_status"] = "unknown"

        ok_ids = set(
            existing_df.loc[
                existing_df["xml_status"].eq("ok"),
                "identificador",
            ].astype(str)
        )
    else:
        existing_df = pd.DataFrame(columns=output_cols)
        ok_ids = set()

    candidates_to_process = candidates[
        ~candidates["identificador"].isin(ok_ids)
    ].copy()

    if candidates_to_process.empty:
        final_df = existing_df.copy()

        for col in output_cols:
            if col not in final_df.columns:
                final_df[col] = None

        final_df = (
            final_df[output_cols]
            .drop_duplicates(subset=["identificador"], keep="last")
            .sort_values(["fecha_publicacion", "identificador"])
            .reset_index(drop=True)
        )

        final_df.to_parquet(output_path, index=False)
        return final_df

    records = []
    parsed_at = datetime.now(timezone.utc).isoformat()

    for row in candidates_to_process.itertuples(index=False):
        xml_path = xml_dir / f"{row.doc_file_stem}.xml"

        base_record = {
            "identificador": row.identificador,
            "doc_file_stem": row.doc_file_stem,
            "url_html": row.url_html,
            "url_xml": row.url_xml,
            "fecha_publicacion": row.fecha_publicacion,
            "titulo": row.titulo,
            "epigrafe_nombre": row.epigrafe_nombre,
            "departamento_nombre": row.departamento_nombre,
            "seccion_nombre": row.seccion_nombre,
            "xml_path": str(xml_path),
            "parsed_at": parsed_at,
        }

        if not xml_path.exists():
            records.append(
                {
                    **base_record,
                    "texto_limpio": "",
                    "texto_len": 0,
                    "xml_status": "missing",
                    "parse_error": "XML file not found",
                }
            )
            continue

        try:
            texto_limpio = parse_boe_xml_to_text(xml_path)

            records.append(
                {
                    **base_record,
                    "texto_limpio": texto_limpio,
                    "texto_len": len(texto_limpio),
                    "xml_status": "ok",
                    "parse_error": None,
                }
            )

        except Exception as exc:
            records.append(
                {
                    **base_record,
                    "texto_limpio": "",
                    "texto_len": 0,
                    "xml_status": "parse_error",
                    "parse_error": repr(exc),
                }
            )

    new_df = pd.DataFrame.from_records(records, columns=output_cols)

    final_df = pd.concat([existing_df, new_df], ignore_index=True)

    for col in output_cols:
        if col not in final_df.columns:
            final_df[col] = None

    final_df = (
        final_df[output_cols]
        .drop_duplicates(subset=["identificador"], keep="last")
        .sort_values(["fecha_publicacion", "identificador"])
        .reset_index(drop=True)
    )

    final_df.to_parquet(output_path, index=False)

    return final_df

# Pruebas

In [ ]:
boe_candidates = pd.read_parquet(BOE_CANDIDATES_PATH)
boe_candidates.loc[boe_candidates["identificador"]=="BOE-A-2024-16664"]

In [ ]:
boe_candidates = pd.read_parquet(BOE_CANDIDATES_PATH)

boe_candidates_docs_text = build_boe_candidates_docs_text(
    candidates=boe_candidates,
    xml_dir=BOE_DOCS_XML_DIR,
    output_path=BOE_CANDIDATES_DOCS_TEXT_PATH,
)

In [ ]:
boe_candidates_docs_text["xml_status"].value_counts(dropna=False)

In [ ]:
boe_candidates_docs_text["texto_limpio"][0]

In [ ]:
boe_candidates_docs_text.columns

In [ ]:
# PE Badulaque
target_ids_badulaque = [
    "BOE-B-2021-32560",
    "BOE-A-2023-2598",
    "BOE-A-2023-10306",
    "BOE-B-2023-19082",
    "BOE-A-2024-16664",
]


In [ ]:
# FV Andévalo
target_ids_andevalo = [
    # Proyecto FV Andévalo e hibridaciones
    "BOE-B-2024-26379",
    "BOE-A-2025-18285",
    "BOE-B-2026-3596",

    # Antecedentes y referencias indirectas
    "BOE-A-2022-24404",  # FV Majal Alto
    "BOE-A-2024-9608",   # FV La Puebla 1
    "BOE-A-2025-26110",  # FV La Puebla 1
    "BOE-A-2026-7629",   # FV La Puebla 3 y 4
    "BOE-A-2026-13454",  # FV La Puebla 3
]

In [ ]:
boe_candidates_docs_text.loc[
    boe_candidates_docs_text["identificador"].isin(target_ids_andevalo)
]
